In [2]:
import time
import traceback
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

def get_meituan_chrome_driver():
    """初始化生产级防爬 Chrome 浏览器"""
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    options.page_load_strategy = 'normal'
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(25)
    
    # 彻底抹除 webdriver 自动化特征
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    return driver

def parse_meituan_detail_page(driver, item_title):
    """解析单个详情页，返回精确的结构化数据字典"""
    job_data = {}
    try:
        # 等待详情页的核心描述区域加载
        WebDriverWait(driver, 12).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '.position_duty, [class*="duty"], [class*="detail"]'))
        )
        time.sleep(1.5)  # 留出充足时间让前端回填异步数据
        
        html_source = driver.page_source
        soup = BeautifulSoup(html_source, 'html.parser')
        
        # --- 职位名称 ---
        title_node = soup.find('div', class_=lambda c: c and 'title' in c)
        job_title = title_node.get_text(strip=True) if title_node else item_title
        
        # --- 工作城市与标签提取 ---
        location = "未知"
        job_type = "未知"
        job_id = "未知"
        
        split_box = soup.find('div', class_=lambda c: c and 'split_line_box' in c)
        if split_box:
            spans = split_box.find_all('span')
            spans_text = [s.get_text(strip=True) for s in spans if s.get_text(strip=True)]
            if len(spans_text) >= 1: location = spans_text[0]
            if len(spans_text) >= 2: job_type = spans_text[1]
            
        # --- 职位描述（岗位职责） ---
        jd_text = "未知"
        job_requirements = "未知"
        
        duty_box = soup.find('div', class_=lambda c: c and 'position_duty' in c)
        if duty_box:
            # 兼容处理：有些结构将描述放在 desc 类中
            desc_node = duty_box.find('div', class_=lambda c: c and 'desc' in c)
            if desc_node:
                jd_text = desc_node.get_text(strip=True)
            else:
                jd_text = duty_box.get_text(strip=True)
                
        # 组装标准字段
        job_data = {
            "职位名称": job_title,
            "工作城市": location,
            "职位类型": job_type,
            "职位描述与要求": jd_text
        }
        print(f"    ✅ 精准解析成功 -> 【{job_title}】({location})")
        
    except Exception as e:
        print(f"    ⚠️ 解析详情页异常: {e}")
        job_data = {
            "职位名称": item_title,
            "工作城市": "未知",
            "职位类型": "未知",
            "职位描述与要求": "解析失败"
        }
    return job_data

def crawl_meituan_one_category(category):
    """批量扫描某一个大类的所有页码，抓取 URL 去重后，再逐个访问详情页精准解析"""
    print(f"\n{'='*60}\n🚀 开始全量扫描美团【{category}】列表页\n{'='*60}")
    
    driver = get_meituan_chrome_driver()
    driver.maximize_window()
    jobs_in_category = []
    all_detail_urls = []
    
    # ------------------- 第一阶段：列表页 URL 批量扫描 -------------------
    try:
        driver.get("https://zhaopin.meituan.com/web/campus")
        time.sleep(5) 
        
        print("1. 正在定位真实的输入框...")
        input_xpath = '//input[@type="text"] | //input[contains(@placeholder, "输入关键词")]'
        real_input = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, input_xpath))
        )
        
        print("2. ⚡ 注入关键词并同步底层状态...")
        driver.execute_script("arguments[0].value = '';", real_input) 
        driver.execute_script(f"arguments[0].value = '{category}';", real_input) 
        
        # 广播事件通知美团的前端底层数据已更新
        driver.execute_script("arguments[0].dispatchEvent(new Event('input', { bubbles: true }));", real_input)
        driver.execute_script("arguments[0].dispatchEvent(new Event('change', { bubbles: true }));", real_input)
        time.sleep(1)
        
        real_input.send_keys(Keys.ENTER)
        print("🎯 已执行回车检索，等待结果列表渲染...")
        time.sleep(5) 
        
        current_page = 1
        while True:
            print(f"  📖 正在扫描第 {current_page} 页列表职位链接...")
            
            # 定位卡片根节点
            cards = driver.find_elements(By.CSS_SELECTOR, 'div[class*="position_list_item"]')
            page_urls = []
            
            for card in cards:
                try:
                    # 抓取标题
                    title_el = card.find_element(By.CSS_SELECTOR, '[class*="title"]')
                    title = title_el.text.strip()
                    
                    # 寻找包含 href 的 a 标签（通常包裹在外面或本身就是）
                    try:
                        if card.tag_name == 'a':
                            href = card.get_attribute("href")
                        else:
                            a_tag = card.find_element(By.XPATH, './ancestor::a | .//a')
                            href = a_tag.get_attribute("href")
                    except:
                        # 兜底：如果美团是用 JS 点击跳转而非 a 标签，则尝试提取 jobunionid 拼接
                        job_id = card.get_attribute("data-jobunionid")
                        if job_id:
                            href = f"https://zhaopin.meituan.com/web/campus/detail?jobUnionId={job_id}"
                        else:
                            continue
                            
                    if href and "detail" in href:
                        page_urls.append({"title": title, "url": href})
                except:
                    continue
            
            print(f"    当前页扫描到 {len(page_urls)} 个职位链接")
            all_detail_urls.extend(page_urls)
            
            # 自动化翻页控制
            try:
                next_button_xpath = (
                    '//li[@title="下一页"]'
                    '| //button[contains(@class, "next")]'
                    '| //li[contains(@class, "pagination-next")]'
                )
                next_button = driver.find_elements(By.XPATH, next_button_xpath)
                
                if not next_button:
                    print("  🛑 未找到翻页按钮，列表扫描自然结束。")
                    break
                    
                btn = next_button[0]
                class_attr = btn.get_attribute("class") or ""
                disabled_attr = btn.get_attribute("aria-disabled") or ""
                
                if "disabled" in class_attr or "disabled" in disabled_attr or "item-disabled" in class_attr:
                    print(f"  🛑 已到达最终页，翻页扫描完成。")
                    break
                
                driver.execute_script("arguments[0].click();", btn)
                current_page += 1
                time.sleep(5)  # 等待下一页彻底渲染
                
            except Exception as page_e:
                print(f"  🛑 翻页遇到异常，停止列表扫描: {page_e}")
                break
                
    except Exception as e:
        print(f"  ❌ 列表扫描阶段发生异常: {e}")
        traceback.print_exc()

    # 全局无死角去重 URL
    seen_urls = set()
    unique_all_urls = [item for item in all_detail_urls if not (item["url"] in seen_urls or seen_urls.add(item["url"]))]
    total_urls = len(unique_all_urls)
    
    print(f"\n📊 【{category}】列表检索完毕，去重后共计获得 {total_urls} 个待解析的独立 URL")
    
    # ------------------- 第二阶段：精细化逐个访问 URL -------------------
    if total_urls > 0:
        print(f"🔍 启动详情页路由切换，正在逐个深入抓取...")
        for idx, item in enumerate(unique_all_urls, 1):
            detail_url = item["url"]
            print(f"  📌 [{idx}/{total_urls}] 正在访问: {detail_url}")
            
            # 详情页重试防丢机制
            retry_count = 2
            success = False
            for retry in range(retry_count):
                try:
                    driver.get(detail_url)
                    job_data = parse_meituan_detail_page(driver, item["title"])
                    job_data["详情页URL"] = detail_url
                    job_data["搜索大类"] = category
                    jobs_in_category.append(job_data)
                    success = True
                    break
                except Exception as visit_e:
                    print(f"    ⚠️ 第 {retry+1} 次载入失败，正在重试... (原因: {visit_e})")
                    time.sleep(3)
            
            if not success:
                print(f"    ❌ 网页加载超时，跳过该职位。")
                jobs_in_category.append({
                    "职位名称": item["title"], "工作城市": "未知", "职位类型": "未知",
                    "职位描述与要求": "网页加载失败", "详情页URL": detail_url, "搜索大类": category
                })
            
            # 详情页之间加入 2~3 秒的随机微休眠，保护 IP 不被封禁
            time.sleep(random.uniform(2.0, 3.5))
            
    driver.quit()
    print(f"🔒 【{category}】数据抓取完成，浏览器驱动安全释放。")
    return jobs_in_category


if __name__ == "__main__":
    # 四大核心校招类别
    target_categories = ["数据分析", "数据科学", "算法", "风控"]
    all_final_jobs = []
    
    for cat in target_categories:
        cat_jobs = crawl_meituan_one_category(cat)
        all_final_jobs.extend(cat_jobs)
        print("\n🍵 大类之间建立风控防线，安全歇息 6 秒...")
        time.sleep(6)
    
    # 保存数据落地
    if all_final_jobs:
        df = pd.DataFrame(all_final_jobs)
        # 按照详情页 URL 去重
        df.drop_duplicates(subset=['详情页URL'], inplace=True)
        
        output_filename = "meituan_campus_detailed_data.csv"
        df.to_csv(output_filename, index=False, encoding="utf-8-sig")
        
        print(f"\n{'='*60}\n🎉 任务圆满成功！\n💾 本次行动共清洗并捕获 {len(df)} 条核心职位明细数据")
        print(f"💾 数据已妥善保存至本地: {output_filename}\n{'='*60}")
    else:
        print("\n❌ 未能获取到任何数据，请确认在当前环境下是否触发了滑动验证码。")


🚀 开始全量扫描美团【数据分析】列表页
1. 正在定位真实的输入框...
2. ⚡ 注入关键词并同步底层状态...
  ❌ 列表扫描阶段发生异常: Message: element not interactable
  (Session info: chrome=148.0.7778.179); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementnotinteractableexception
Stacktrace:
0   chromedriver                        0x0000000102f43e68 chromedriver + 6856296
1   chromedriver                        0x0000000102f3b4da chromedriver + 6821082
2   chromedriver                        0x0000000102928880 chromedriver + 452736
3   chromedriver                        0x0000000102977f51 chromedriver + 778065
4   chromedriver                        0x0000000102975ee8 chromedriver + 769768
5   chromedriver                        0x0000000102971728 chromedriver + 751400
6   chromedriver                        0x00000001029c5786 chromedriver + 1095558
7   chromedriver                        0x000000010296fef1 chromedriver + 745201
8   chromedriver                    

Traceback (most recent call last):
  File "/var/folders/bv/6c0cxtgs0xgbcxf8ynl_f8zh0000gn/T/ipykernel_70644/2018617156.py", line 126, in crawl_meituan_one_category
    real_input.send_keys(Keys.ENTER)
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/webelement.py", line 276, in send_keys
    self._execute(
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/webelement.py", line 508, in _execute
    return self._parent.execute(command, params)
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/webdriver.py", line 468, in execute
    self.error_handler.check_response(response)
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/errorhandler.py", line 232, in check_response
    raise exception_class(message, screen, stacktrace)
selenium.common.exceptions.El


🚀 开始全量扫描美团【数据科学】列表页
1. 正在定位真实的输入框...
2. ⚡ 注入关键词并同步底层状态...
  ❌ 列表扫描阶段发生异常: Message: element not interactable
  (Session info: chrome=148.0.7778.179); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementnotinteractableexception
Stacktrace:
0   chromedriver                        0x0000000100828e68 chromedriver + 6856296
1   chromedriver                        0x00000001008204da chromedriver + 6821082
2   chromedriver                        0x000000010020d880 chromedriver + 452736
3   chromedriver                        0x000000010025cf51 chromedriver + 778065
4   chromedriver                        0x000000010025aee8 chromedriver + 769768
5   chromedriver                        0x0000000100256728 chromedriver + 751400
6   chromedriver                        0x00000001002aa786 chromedriver + 1095558
7   chromedriver                        0x0000000100254ef1 chromedriver + 745201
8   chromedriver                    

Traceback (most recent call last):
  File "/var/folders/bv/6c0cxtgs0xgbcxf8ynl_f8zh0000gn/T/ipykernel_70644/2018617156.py", line 126, in crawl_meituan_one_category
    real_input.send_keys(Keys.ENTER)
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/webelement.py", line 276, in send_keys
    self._execute(
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/webelement.py", line 508, in _execute
    return self._parent.execute(command, params)
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/webdriver.py", line 468, in execute
    self.error_handler.check_response(response)
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/errorhandler.py", line 232, in check_response
    raise exception_class(message, screen, stacktrace)
selenium.common.exceptions.El


🚀 开始全量扫描美团【算法】列表页
1. 正在定位真实的输入框...
2. ⚡ 注入关键词并同步底层状态...
  ❌ 列表扫描阶段发生异常: Message: element not interactable
  (Session info: chrome=148.0.7778.179); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementnotinteractableexception
Stacktrace:
0   chromedriver                        0x0000000100fb4e68 chromedriver + 6856296
1   chromedriver                        0x0000000100fac4da chromedriver + 6821082
2   chromedriver                        0x0000000100999880 chromedriver + 452736
3   chromedriver                        0x00000001009e8f51 chromedriver + 778065
4   chromedriver                        0x00000001009e6ee8 chromedriver + 769768
5   chromedriver                        0x00000001009e2728 chromedriver + 751400
6   chromedriver                        0x0000000100a36786 chromedriver + 1095558
7   chromedriver                        0x00000001009e0ef1 chromedriver + 745201
8   chromedriver                      

Traceback (most recent call last):
  File "/var/folders/bv/6c0cxtgs0xgbcxf8ynl_f8zh0000gn/T/ipykernel_70644/2018617156.py", line 126, in crawl_meituan_one_category
    real_input.send_keys(Keys.ENTER)
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/webelement.py", line 276, in send_keys
    self._execute(
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/webelement.py", line 508, in _execute
    return self._parent.execute(command, params)
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/webdriver.py", line 468, in execute
    self.error_handler.check_response(response)
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/errorhandler.py", line 232, in check_response
    raise exception_class(message, screen, stacktrace)
selenium.common.exceptions.El


🚀 开始全量扫描美团【风控】列表页
1. 正在定位真实的输入框...
2. ⚡ 注入关键词并同步底层状态...
  ❌ 列表扫描阶段发生异常: Message: element not interactable
  (Session info: chrome=148.0.7778.179); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementnotinteractableexception
Stacktrace:
0   chromedriver                        0x0000000100b26e68 chromedriver + 6856296
1   chromedriver                        0x0000000100b1e4da chromedriver + 6821082
2   chromedriver                        0x000000010050b880 chromedriver + 452736
3   chromedriver                        0x000000010055af51 chromedriver + 778065
4   chromedriver                        0x0000000100558ee8 chromedriver + 769768
5   chromedriver                        0x0000000100554728 chromedriver + 751400
6   chromedriver                        0x00000001005a8786 chromedriver + 1095558
7   chromedriver                        0x0000000100552ef1 chromedriver + 745201
8   chromedriver                      

Traceback (most recent call last):
  File "/var/folders/bv/6c0cxtgs0xgbcxf8ynl_f8zh0000gn/T/ipykernel_70644/2018617156.py", line 126, in crawl_meituan_one_category
    real_input.send_keys(Keys.ENTER)
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/webelement.py", line 276, in send_keys
    self._execute(
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/webelement.py", line 508, in _execute
    return self._parent.execute(command, params)
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/webdriver.py", line 468, in execute
    self.error_handler.check_response(response)
  File "/Users/huluobodediannao/anaconda3/envs/nlp2026/lib/python3.10/site-packages/selenium/webdriver/remote/errorhandler.py", line 232, in check_response
    raise exception_class(message, screen, stacktrace)
selenium.common.exceptions.El


❌ 未能获取到任何数据，请确认在当前环境下是否触发了滑动验证码。
